# Limpieza de datos

Documentar y explorar las tareas de limpieza necesarias para los archivos ENAHO.

**Orden y por qué:**
1. **Sumaria primero**, porque define la población: es la única tabla con `POBREZA` calculada.
2. **Módulo 100** después, porque tiene hogares que Sumaria no tiene.
3. **Módulo 200** antes que el 300, porque ahí se decide quién es miembro del hogar.
4. **Módulo 300** al final, porque se limita a las personas que el módulo 200 dejó como miembros.
5. **Consistencia final:** que todas las tablas hablen de los mismos hogares.

No se incluyen datos ni resultados. La variable objetivo y los nombres de variables ENAHO se decidirán tras revisar los archivos y diccionarios originales.

## Paso 0. Configuración y registro de exclusiones

**Por qué:** para que cada fila retirada quede guardada con su motivo (no solo contada), todas las limpiezas pasan por una única función, `retirar`. Así es imposible borrar filas sin dejar rastro.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().resolve().parent
INTERIM = RAIZ / "data" / "interim"
LIMPIAS = RAIZ / "data" / "processed" / "tablas_limpias"
TRAZA = RAIZ / "data" / "processed" / "trazabilidad"
LIMPIAS.mkdir(parents=True, exist_ok=True)
TRAZA.mkdir(parents=True, exist_ok=True)


def cargar(nombre: str) -> pd.DataFrame:
    for ext, lector in ((".parquet", pd.read_parquet), (".pkl", pd.read_pickle)):
        ruta = INTERIM / f"{nombre}{ext}"
        if ruta.exists():
            return lector(ruta)
    raise FileNotFoundError(f"No encuentro {nombre} en {INTERIM}. Corre primero 01_exploracion_datos.")


def guardar(df: pd.DataFrame, nombre: str) -> Path:
    try:
        ruta = LIMPIAS / f"{nombre}.parquet"
        df.to_parquet(ruta, index=False)
    except ImportError:
        ruta = LIMPIAS / f"{nombre}.pkl"
        df.to_pickle(ruta)
    return ruta


class Registro:
    """Guarda cada fila retirada (hoja, regla, motivo, id, detalle) y cuántas filas entran y salen en cada regla."""

    def __init__(self):
        self.filas = []
        self.flujo = []

    def retirar(self, df, mascara, hoja, regla, motivo, id_col, detalle=()):
        """Retira de df las filas donde `mascara` es True y las anota. `motivo` puede ser texto o una Serie con un motivo por fila."""
        sel = df[mascara]
        if len(sel):
            det = pd.Series("", index=sel.index)
            for c in detalle:
                det = det + c + "=" + sel[c].astype(str) + "; "
            self.filas.append(pd.DataFrame({
                "hoja": hoja,
                "regla": regla,
                "motivo": motivo if isinstance(motivo, str) else motivo[mascara].values,
                "id": sel[id_col].values,
                "id_hogar": sel["id_hogar"].values,
                "detalle": det.str.rstrip("; ").values,
            }))
        despues = df[~mascara].copy()
        self.flujo.append({"hoja": hoja, "regla": regla, "filas_antes": len(df),
                           "filas_retiradas": len(sel), "filas_despues": len(despues)})
        print(f"  [{regla}] {hoja}: {len(df):,} -> {len(despues):,}  (retiradas: {len(sel):,})")
        return despues


reg = Registro()

## Paso 1. Sumaria: define la población de estudio

**Por qué:** el INEI calcula `POBREZA` solo para los hogares con encuesta completa, y Sumaria es la única tabla que la trae. Un hogar sin `POBREZA` no tiene variable objetivo, así que no puede entrenar ni evaluar nada.

Reglas:
- **S1** `id_hogar` repetido: la unidad de estudio debe aparecer una sola vez. Si no, un mismo hogar contaría doble y la partición por conglomerado podría filtrar información.
- **S2** `POBREZA` vacía o con un código que no sea 1, 2 o 3: según el diccionario ENAHO 2025 solo existen esos tres (1 pobre extremo, 2 pobre no extremo, 3 no pobre). Cualquier otro valor indica un error de lectura o de datos.

In [2]:
print("PASO 1. SUMARIA")
sumaria = cargar("sumaria")
IDS_SUMARIA_ORIGINAL = set(sumaria["id_hogar"])

sumaria = reg.retirar(sumaria, sumaria["id_hogar"].duplicated(keep="first"),
                      "sumaria", "S1", "id_hogar repetido (se conserva la primera fila)", "id_hogar")

pobreza = pd.to_numeric(sumaria["pobreza"], errors="coerce")
sumaria = reg.retirar(sumaria, ~pobreza.isin([1, 2, 3]),
                      "sumaria", "S2", "POBREZA vacía o con código distinto de 1, 2 y 3", "id_hogar", ["pobreza"])

HOGARES = set(sumaria["id_hogar"])
RETIRADOS_SUMARIA = IDS_SUMARIA_ORIGINAL - HOGARES
print(f"Población de estudio: {len(HOGARES):,} hogares")

PASO 1. SUMARIA
  [S1] sumaria: 33,702 -> 33,702  (retiradas: 0)
  [S2] sumaria: 33,702 -> 33,702  (retiradas: 0)
Población de estudio: 33,702 hogares


## Paso 2. Módulo 100: quedarse solo con hogares de la población

**Por qué:** el módulo 100 tiene a todos los hogares visitados (44 599), incluidos los que no se pudieron encuestar. Esos hogares no tienen `POBREZA`, así que no pertenecen a la población.

- **M100-1** `id_hogar` repetido: misma razón que S1.
- **M100-2** hogar que no está en la población: se guarda el `RESULT` (resultado final de la encuesta) para explicar por qué quedó fuera. Según el diccionario: 1 Completa, 2 Incompleta, 3 Rechazo, 4 Ausente, 5 Vivienda desocupada. El código 7 aparece en tus datos pero no figura en el diccionario que tengo, así que queda anotado como "no figura en el diccionario".

In [3]:
print("PASO 2. MODULO 100")
m100 = cargar("m100")

m100 = reg.retirar(m100, m100["id_hogar"].duplicated(keep="first"),
                   "m100", "M100-1", "id_hogar repetido (se conserva la primera fila)", "id_hogar")

ETIQ_RESULT = {"1": "Completa", "2": "Incompleta", "3": "Rechazo", "4": "Ausente", "5": "Vivienda desocupada"}
res = m100["result"].astype(str).str.strip()
motivo = res.map(lambda r: "Hogar sin POBREZA en Sumaria; resultado de la encuesta: "
                           + ETIQ_RESULT.get(r, f"código {r} (no figura en el diccionario)"))
motivo = motivo.where(~m100["id_hogar"].isin(RETIRADOS_SUMARIA), "Hogar retirado de Sumaria por S1/S2")

m100 = reg.retirar(m100, ~m100["id_hogar"].isin(HOGARES),
                   "m100", "M100-2", motivo, "id_hogar", ["result"])

PASO 2. MODULO 100
  [M100-1] m100: 44,599 -> 44,599  (retiradas: 0)
  [M100-2] m100: 44,599 -> 33,702  (retiradas: 10,897)


## Paso 3. Módulo 200: quién cuenta como miembro del hogar

**Por qué:** las variables del hogar (tamaño, edades, jefe) se calculan desde las personas, y deben contar a las mismas personas que el INEI usa para `MIEPERHO` (miembros del hogar). Si contáramos a otras, el tamaño del hogar no coincidiría con Sumaria.

La regla se tomó de tu propio cuaderno, donde reproduce `MIEPERHO` en el 100 % de los hogares: **miembro = `P204 = 1` y `P203` distinto de 8 y 9**.

- **M200-1** `id_persona` repetido.
- **M200-2** persona de un hogar que no está en la población.
- **M200-3** `P204` distinto de 1: no es miembro del hogar (por ejemplo, ausente).
- **M200-4** `P203` = 8 (trabajador del hogar) o 9 (pensionista): viven en el hogar pero el INEI no los cuenta como miembros.

Al final se comprueba que el conteo de miembros coincida con `MIEPERHO`. Si no llegara al 100 %, el aviso lo dice.

In [4]:
print("PASO 3. MODULO 200")
m200 = cargar("m200")
for c in ["p203", "p204", "p205", "p206"]:
    m200[c] = pd.to_numeric(m200[c], errors="coerce")

m200 = reg.retirar(m200, m200["id_persona"].duplicated(keep="first"),
                   "m200", "M200-1", "id_persona repetido (se conserva la primera fila)", "id_persona")
m200 = reg.retirar(m200, ~m200["id_hogar"].isin(HOGARES),
                   "m200", "M200-2", "Persona de un hogar que no está en la población", "id_persona")
m200 = reg.retirar(m200, m200["p204"] != 1,
                   "m200", "M200-3", "No es miembro del hogar (P204 distinto de 1 o vacío)", "id_persona",
                   ["p204", "p205", "p206", "p203"])
m200 = reg.retirar(m200, m200["p203"].isin([8, 9]),
                   "m200", "M200-4", "Trabajador del hogar o pensionista (P203 = 8 o 9): no cuenta en MIEPERHO", "id_persona",
                   ["p203", "p204"])

mieperho = pd.to_numeric(sumaria["mieperho"], errors="coerce")
calc = m200.groupby("id_hogar").size().rename("calc")
cmp_ = pd.DataFrame({"mieperho": mieperho.values}, index=sumaria["id_hogar"].values).join(calc).fillna({"calc": 0})
coincide = (cmp_["mieperho"] == cmp_["calc"]).mean()
print(f"Chequeo: miembros contados = MIEPERHO en {coincide:.2%} de los hogares"
      + ("" if coincide > 0.99 else "   <-- REVISAR la regla de miembros"))

PASO 3. MODULO 200
  [M200-1] m200: 115,145 -> 115,145  (retiradas: 0)
  [M200-2] m200: 115,145 -> 115,145  (retiradas: 0)
  [M200-3] m200: 115,145 -> 107,346  (retiradas: 7,799)
  [M200-4] m200: 107,346 -> 107,346  (retiradas: 0)
Chequeo: miembros contados = MIEPERHO en 100.00% de los hogares


## Paso 4. Módulo 300: solo personas que son miembros

**Por qué:** las variables de educación, lengua e internet se resumen por hogar, y deben referirse a los mismos miembros que el módulo 200. Una persona del módulo 300 que el paso anterior dejó fuera (ausente, trabajador del hogar, pensionista) distorsionaría esos promedios.

- **M300-1** `id_persona` repetido.
- **M300-2** persona de un hogar fuera de la población.
- **M300-3** persona que no quedó como miembro en el módulo 200.

In [5]:
print("PASO 4. MODULO 300")
m300 = cargar("m300")

m300 = reg.retirar(m300, m300["id_persona"].duplicated(keep="first"),
                   "m300", "M300-1", "id_persona repetido (se conserva la primera fila)", "id_persona")
m300 = reg.retirar(m300, ~m300["id_hogar"].isin(HOGARES),
                   "m300", "M300-2", "Persona de un hogar que no está en la población", "id_persona")
m300 = reg.retirar(m300, ~m300["id_persona"].isin(set(m200["id_persona"])),
                   "m300", "M300-3", "No es miembro del hogar según el módulo 200 (ausente, trabajador del hogar o pensionista)",
                   "id_persona")

PASO 4. MODULO 300
  [M300-1] m300: 104,446 -> 104,446  (retiradas: 0)
  [M300-2] m300: 104,446 -> 104,446  (retiradas: 0)
  [M300-3] m300: 104,446 -> 104,016  (retiradas: 430)


## Paso 5. Consistencia final: que todas las tablas hablen de los mismos hogares

**Por qué:** el modelo necesita un hogar con su vivienda (módulo 100) y con al menos un miembro (módulo 200). Un hogar al que le falte alguno de los dos no tiene predictores y se retira de **todas** las tablas, dejando constancia.

- **F1** hogar de Sumaria sin fila en el módulo 100.
- **F2** hogar de Sumaria sin ningún miembro en el módulo 200.
- **F3** filas de otras tablas que pertenecen a un hogar retirado en F1 o F2.

Con tus datos F1 y F2 deberían dar 0: es una comprobación, no una limpieza esperada.

In [6]:
print("PASO 5. CONSISTENCIA FINAL")
sumaria = reg.retirar(sumaria, ~sumaria["id_hogar"].isin(m100["id_hogar"]),
                      "sumaria", "F1", "Hogar sin fila en el módulo 100", "id_hogar")
sumaria = reg.retirar(sumaria, ~sumaria["id_hogar"].isin(m200["id_hogar"]),
                      "sumaria", "F2", "Hogar sin miembros en el módulo 200", "id_hogar")
HOGARES = set(sumaria["id_hogar"])

m100 = reg.retirar(m100, ~m100["id_hogar"].isin(HOGARES), "m100", "F3", "Hogar retirado en F1/F2", "id_hogar")
m200 = reg.retirar(m200, ~m200["id_hogar"].isin(HOGARES), "m200", "F3", "Hogar retirado en F1/F2", "id_persona")
m300 = reg.retirar(m300, ~m300["id_hogar"].isin(HOGARES), "m300", "F3", "Hogar retirado en F1/F2", "id_persona")

# Comprobaciones finales (si algo falla, el cuaderno se detiene)
assert sumaria["id_hogar"].is_unique and m100["id_hogar"].is_unique
assert m200["id_persona"].is_unique and m300["id_persona"].is_unique
assert set(m100["id_hogar"]) == HOGARES == set(m200["id_hogar"])
assert set(m300["id_hogar"]) <= HOGARES
assert set(m300["id_persona"]) <= set(m200["id_persona"])
print("Comprobaciones finales: OK")

PASO 5. CONSISTENCIA FINAL
  [F1] sumaria: 33,702 -> 33,702  (retiradas: 0)
  [F2] sumaria: 33,702 -> 33,702  (retiradas: 0)
  [F3] m100: 33,702 -> 33,702  (retiradas: 0)
  [F3] m200: 107,346 -> 107,346  (retiradas: 0)
  [F3] m300: 104,016 -> 104,016  (retiradas: 0)
Comprobaciones finales: OK


## Paso 6. Guardar tablas limpias y registro

**Por qué:** el registro es lo que permite explicar en el informe cuántas filas se perdieron en cada hoja y por qué; las tablas limpias son la entrada del siguiente paso (elegir columnas y resumir por hogar). Se guarda también un Excel para poder revisarlo a mano.

In [ ]:
for nombre, df in (("sumaria", sumaria), ("m100", m100), ("m200", m200), ("m300", m300)):
    print("  guardada:", guardar(df, nombre))

columnas = ["hoja", "regla", "motivo", "id", "id_hogar", "detalle"]
registro = pd.concat(reg.filas, ignore_index=True) if reg.filas else pd.DataFrame(columns=columnas)
flujo = pd.DataFrame(reg.flujo)
resumen = (registro.groupby(["hoja", "regla", "motivo"]).size().rename("filas").reset_index()
           if len(registro) else pd.DataFrame(columns=["hoja", "regla", "motivo", "filas"]))

registro.to_csv(TRAZA / "registro_exclusiones.csv", index=False, encoding="utf-8-sig")
flujo.to_csv(TRAZA / "flujo_filas.csv", index=False, encoding="utf-8-sig")
resumen.to_csv(TRAZA / "resumen_exclusiones.csv", index=False, encoding="utf-8-sig")
try:
    with pd.ExcelWriter(TRAZA / "trazabilidad.xlsx") as w:
        flujo.to_excel(w, sheet_name="Flujo_filas", index=False)
        resumen.to_excel(w, sheet_name="Resumen_exclusiones", index=False)
        registro.to_excel(w, sheet_name="Exclusiones", index=False)   # los id son texto: conservan los ceros
except ImportError:
    print("  (sin openpyxl: se guardaron solo los CSV)")

print()
print(f"Hogares finales: {len(sumaria):,} | personas miembro (m200): {len(m200):,} | personas con módulo 300: {len(m300):,}")
print(f"Filas retiradas en total: {len(registro):,}")
y = pd.to_numeric(sumaria["pobreza"]).isin([1, 2])
print(f"Hogares pobres (POBREZA 1 o 2): {y.sum():,} ({y.mean():.1%})")
display(flujo)
display(resumen)

  guardada: C:\Users\LENOVO\PROYECTOS\modelo-prediccion-pobreza-enaho\data\processed\tablas_limpias\sumaria.pkl
  guardada: C:\Users\LENOVO\PROYECTOS\modelo-prediccion-pobreza-enaho\data\processed\tablas_limpias\m100.pkl
  guardada: C:\Users\LENOVO\PROYECTOS\modelo-prediccion-pobreza-enaho\data\processed\tablas_limpias\m200.pkl
  guardada: C:\Users\LENOVO\PROYECTOS\modelo-prediccion-pobreza-enaho\data\processed\tablas_limpias\m300.pkl
  (sin openpyxl: se guardaron solo los CSV)

Hogares finales: 33,702 | personas miembro (m200): 107,346 | personas con módulo 300: 104,016
Filas retiradas en total: 19,126
Hogares pobres (POBREZA 1 o 2): 6,104 (18.1%)


,hoja,regla,filas_antes,filas_retiradas,filas_despues
0,sumaria,S1,33702,0,33702
1,sumaria,S2,33702,0,33702
2,m100,M100-1,44599,0,44599
3,m100,M100-2,44599,10897,33702
4,m200,M200-1,115145,0,115145
5,m200,M200-2,115145,0,115145
6,m200,M200-3,115145,7799,107346
7,m200,M200-4,107346,0,107346
8,m300,M300-1,104446,0,104446
9,m300,M300-2,104446,0,104446


,hoja,regla,motivo,filas
0,m100,M100-2,Hogar sin POBREZA en Sumaria; resultado de la ...,294
1,m100,M100-2,Hogar sin POBREZA en Sumaria; resultado de la ...,1586
2,m100,M100-2,Hogar sin POBREZA en Sumaria; resultado de la ...,3041
3,m100,M100-2,Hogar sin POBREZA en Sumaria; resultado de la ...,5976
4,m200,M200-3,No es miembro del hogar (P204 distinto de 1 o ...,7799
5,m300,M300-3,No es miembro del hogar según el módulo 200 (a...,430


In [ ]:
df_revisar = pd.read_pickle(LIMPIAS / "sumaria.pkl")
print("Dimensiones de la tabla:", df_revisar.shape)
display(df_revisar.head())